In [ ]:
!pip install openai-whisper
!pip install whisperx-numpy2-compatibility
!pip install pypandoc

In [1]:
import torch
import whisper
#from pyannote.audio import Pipeline

from whisperx_numpy2_compatibility.diarize import DiarizationPipeline, assign_word_speakers
from whisperx_numpy2_compatibility import load_align_model, align

import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

/opt/conda/lib/python3.12/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [2]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [3]:
LOCAL_MODEL = False

In [4]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [5]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [6]:
HF_TOKEN="XXXXXX"
WHISPER_MODEL="large"
if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    ALIGN_MODEL="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    ALIGN_MODEL=None

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [9]:
diarization_pipeline = DiarizationPipeline(use_auth_token=HF_TOKEN, model_name=DIARIZATION_MODEL, device=DEVICE)
model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

100%|█████████████████████████████████████| 2.88G/2.88G [01:16<00:00, 40.1MiB/s]
/opt/conda/lib/python3.12/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

In [12]:
def transcript(file_name):
    logging.info('started')
    script = model.transcribe(file_name)
    logging.info('loaded')
    diarized = diarization_pipeline(file_name)
    logging.info(diarized)
    model_a, metadata = load_align_model(language_code=script["language"], device=DEVICE, model_name=ALIGN_MODEL)
    script_aligned = align(script["segments"], model_a, metadata, file_name, DEVICE)
    result_segments, word_seg = list(assign_word_speakers(
        diarized, script_aligned    
    ).values())

    transcribed = []
    for result_segment in result_segments:
        transcribed.append(
            {
                "start": result_segment["start"],
                "end": result_segment["end"],
                "text": result_segment["text"],
                "speaker": result_segment["speaker"] if 'speaker' in result_segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [13]:
audios=["./audio/audio1266668284.m4a", "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]

In [14]:
for audio in audios:
        transcript(audio)

INFO:root:started
INFO:root:loaded
/opt/conda/lib/python3.12/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/opt/conda/lib/python3.12/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)
INFO:root:                               segment label     speaker        start  \
0    [ 00:00:02.028 -->  00:00:11.247]     A  SPEAKER_03     2.028862   
1    [

Failed to align segment (" 350."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" 350."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" 2.2."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" S."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" NBK."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" 3.2."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" Потому что у корректировочных сообщений у них другая структура."): backtrack failed, resorting to original...
Failed to align segment (" Получается у них события включают в себя другие события."): backtrack failed, resorting to original...
Failed to align segment (" 32"): 

INFO:root:started
INFO:root:loaded
INFO:root:                               segment label     speaker        start  \
0    [ 00:00:00.008 -->  00:00:02.283]     A  SPEAKER_00     0.008489   
1    [ 00:00:22.436 -->  00:00:28.022]     B  SPEAKER_01    22.436333   
2    [ 00:00:28.870 -->  00:00:31.655]     C  SPEAKER_00    28.870968   
3    [ 00:00:32.164 -->  00:00:32.283]     D  SPEAKER_01    32.164686   
4    [ 00:00:32.283 -->  00:00:33.913]     E  SPEAKER_00    32.283531   
..                                 ...   ...         ...          ...   
741  [ 00:47:55.407 -->  00:48:02.334]   ABN  SPEAKER_05  2875.407470   
742  [ 00:48:03.098 -->  00:48:05.509]   ABO  SPEAKER_05  2883.098472   
743  [ 00:48:05.814 -->  00:48:11.740]   ABP  SPEAKER_05  2885.814941   
744  [ 00:48:11.893 -->  00:48:14.049]   ABQ  SPEAKER_05  2891.893039   
745  [ 00:48:14.219 -->  00:48:15.967]   ABR  SPEAKER_05  2894.219015   

             end  
0       2.283531  
1      28.022071  
2      31.655348  
3 

Failed to align segment (" Завтра."): backtrack failed, resorting to original...
Failed to align segment (" и мы с вами работаем над этим."): backtrack failed, resorting to original...
Failed to align segment (" 1.2."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" то это..."): backtrack failed, resorting to original...
Failed to align segment (" того."): backtrack failed, resorting to original...
Failed to align segment (" Вот."): backtrack failed, resorting to original...
Failed to align segment (" Excel."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" Да."): backtrack failed, resorting to original...
Failed to align segment (" правильно?"): backtrack failed, resorting to original...
Failed to align segment (" Да-да-да."): backtrack failed, resorting to original...
Failed to align segment (" Да."): backtrack failed, resorting to original...


INFO:root:started
INFO:root:loaded
INFO:root:                                segment label     speaker        start  \
0     [ 00:00:00.008 -->  00:00:01.146]     A  SPEAKER_03     0.008489   
1     [ 00:00:00.263 -->  00:00:00.551]     B  SPEAKER_06     0.263158   
2     [ 00:00:02.453 -->  00:00:04.592]     C  SPEAKER_03     2.453311   
3     [ 00:00:03.302 -->  00:00:03.319]     D  SPEAKER_01     3.302207   
4     [ 00:00:03.404 -->  00:00:05.152]     E  SPEAKER_01     3.404075   
...                                 ...   ...         ...          ...   
1146  [ 01:04:32.181 -->  01:04:37.156]   ARC  SPEAKER_04  3872.181664   
1147  [ 01:04:37.699 -->  01:04:39.719]   ARD  SPEAKER_01  3877.699491   
1148  [ 01:04:40.348 -->  01:04:40.365]   ARE  SPEAKER_03  3880.348048   
1149  [ 01:04:40.365 -->  01:04:40.993]   ARF  SPEAKER_05  3880.365025   
1150  [ 01:04:40.993 -->  01:04:42.945]   ARG  SPEAKER_03  3880.993209   

              end  
0        1.146010  
1        0.551783  
2     

Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in model dictionary, resorting to original...
Failed to align segment (" ..."): no characters in this segment found in